In [ ]:
import pandas as pd
from sklearn.preprocessing import minmax_scale

In [ ]:
df = pd.read_csv("final_transformations/F2_SAMPLE_DATA_WITH_TIMESTAMP.csv")
# df_mae_p = pd.read_csv("mae_producto_20rows.csv")
"""
Columns
= alternative
== same
1. Tipo subestrategia == DES_TIPO_SUBESTRATEGIA
2. Tipo grupo == DES_TIPO_GRUPO
3. Indicator padre == ES_PADRE
4. Indicator gratis = ES_GRATIS
5. Factor of repetition == FACTOR_REPETICION

"""

In [ ]:
print(df.shape)
df_filtered = df[df["CODCUC"] != "XXXXXXXXX"]
# print(df_filtered)
print(df_filtered.shape)

In [ ]:
df_filtered["Composite_key"] = df_filtered[["DES_TIPO_SUBESTRATEGIA",group_type,codcuc,is_father,is_free,factor_repitation]].astype(str).agg('|'.join, axis=1)

In [ ]:
df_filtered = df_filtered.drop("COMPOSITE_PRIMARY_KEY", axis=1)  # Remove "Column2"


In [ ]:
def cal_recency(ref_date, last_purchase_date):
    last_purchase_date = pd.Timestamp(last_purchase_date)

    # Calculate the difference in days
    days_difference = (ref_date - last_purchase_date).days

    # Invert the difference for recency
    recency = 1 / (days_difference + 1)  
    return recency

# Reference date
reference_date = pd.Timestamp('2024-12-29') # We can also use system date by doing time.time()

# Calculate recency
recency_values = [cal_recency(reference_date, date) for date in df_filtered['FECHAPROCESO']]

df_filtered["recency"] = recency_values

df_filtered.to_csv("final_transformations/zero_phase.csv", index=False)

print(df_filtered)

In [ ]:
concatenateds = df_filtered.groupby("ID_OFERTA")[["Composite_key", "CODEBELISTA", "recency"]].agg(
    {
        'CODEBELISTA': lambda x: x.dropna().tolist(),
        'Composite_key': lambda x: '|'.join(x),
        'recency': 'mean'  # Aggregating recency by mean
    }
).reset_index()
# Expand the column C into separate rows
expanded_df = concatenateds.explode('CODEBELISTA')

print(expanded_df)
# ave DataFrame to a CSV file
expanded_df.to_csv("final_transformations/second_phase.csv", index=False)
# print(concatenateds.shape)

In [ ]:
# Group by CODEBELISTA and Composite_key, aggregating ID_OFERTA into a list and calculating count
grouped_df = expanded_df.groupby(['CODEBELISTA', 'Composite_key']).agg({
    'ID_OFERTA': lambda x: x.tolist(),  # convert ID_OFERTA to list,
    'recency':'mean'
}).reset_index()

# Add the count column
grouped_df['count'] = expanded_df.groupby(['CODEBELISTA', 'Composite_key']).size().values

# Normalize the 'value' column to be between 0 and 1
grouped_df['normalized_count'] = minmax_scale(grouped_df['count'])
grouped_df['normalized_recency'] = minmax_scale(grouped_df['recency'])

print(grouped_df)

grouped_df.to_csv("final_transformations/third_phase.csv", index=False)

In [ ]:

#Score Calculation
def score_calculation(recency,frequency,weight1=0.3,weight2=0.7):
    score = weight1*recency + frequency*weight2
    return score

grouped_df['score'] = grouped_df.apply(lambda row: score_calculation(row['normalized_recency'], row['normalized_count']), axis=1)

# Sorting score in descending order
sorted_df = grouped_df.sort_values('score', ascending=False)

print(grouped_df)

sorted_df.to_csv("final_transformations/fourth_phase.csv", index=False)
